# PN26 — dominant-parent ridge locator

## TL;DR

One complete lower Phase A parent located the exact next prime on **93.983%** of 6,000 prospectively frozen
anchors. The next prime was in its first two quiet states on **99.650%** and first three on **99.967%**. Three ARA
thresholds passed; a deliberately severe 50-point control-margin failed, so the frozen status is **partial
dominant-parent support**. The fixed `3.5` route is an exact scale frame with zero predictive variance.


## Context & Methods

For each narrow scale cohort, children through `sqrt(2S)` are split at the cumulative-log half. Phase A retains
the smaller, frequent gates and Phase B retains the larger, sparse gates. The primary script—without a primality
test—sealed the first three Phase A quiet candidates at each arbitrary anchor. An independent full segmented-prime
mask then revealed the true next-prime rank.

### Key assumptions

- The cohort lower boundary `S` declares the local rung `S -> 2S`.
- The log-half split is frozen from PN19 and is not refitted on target results.
- Visible ranked states are not arithmetic-operation counts; Phase A still contains many prime children.
- Validator v1.1 changes only a documented prime-table ceiling bug. The sealed predictions are unchanged.


In [1]:
import csv
import json
from pathlib import Path

HERE = Path.cwd()
primary = json.loads((HERE / 'PN26_DOMINANT_PARENT_RIDGE_PRIMARY.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN26_DOMINANT_PARENT_RIDGE_VALIDATION_V1_1.json').read_text(encoding='utf-8'))
failed_v1 = json.loads((HERE / 'PN26_DOMINANT_PARENT_RIDGE_VALIDATION.json').read_text(encoding='utf-8'))
with (HERE / 'PN26_DOMINANT_PARENT_RIDGE_VALIDATED_ROWS_V1_1.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print(primary['status'])
print(validation['status'], validation['checks_passed'], '/', validation['checks_total'])
print('preserved v1 status:', failed_v1['status'])
assert primary['row_count'] == 6000
assert len(rows) == 6000
assert validation['checks_passed'] == validation['checks_total'] == 16


PREDICTIONS SEALED; TARGET TRUTH UNOPENED BY PRIMARY
PARTIAL DOMINANT-PARENT SUPPORT 16 / 16
preserved v1 status: IMPLEMENTATION FAILURE


## Data

In [2]:
for metadata in primary['scale_metadata']:
    print(metadata)
assert primary['protected_87_bit_anchor_used'] is False
assert validation['protected_87_bit_anchor_used'] is False
assert all(sum(row['cohort'] == cohort for row in rows) == 2000 for cohort in ('low','middle','high'))


{'cohort': 'low', 'scale_anchor': 71000000, 'child_limit': 11916, 'child_count': 1427, 'split_index': 780, 'phase_a_count': 780, 'phase_b_count': 647, 'phase_a_last_child': 5939, 'phase_b_first_child': 5953, 'teara_phase_a': 0.999762684493492, 'teara_phase_b': 1.0002373155065079}
{'cohort': 'middle', 'scale_anchor': 71000000000, 'child_limit': 376828, 'child_count': 32055, 'split_index': 17045, 'phase_a_count': 17045, 'phase_b_count': 15010, 'phase_a_last_child': 188579, 'phase_b_first_child': 188603, 'teara_phase_a': 1.0000295336164393, 'teara_phase_b': 0.9999704663835607}
{'cohort': 'high', 'scale_anchor': 710000000000, 'child_limit': 1191637, 'child_count': 92341, 'split_index': 48817, 'phase_a_count': 48817, 'phase_b_count': 43524, 'phase_a_last_child': 596179, 'phase_b_first_child': 596209, 'teara_phase_a': 1.000007356619666, 'teara_phase_b': 0.9999926433803339}


## Results — prospective ranked locator

In [3]:
for summary in validation['summaries']:
    print(
        summary['cohort'],
        'top1=', round(summary['phase_a_top1_rate'], 6),
        'top2=', round(summary['phase_a_top2_rate'], 6),
        'top3=', round(summary['phase_a_top3_rate'], 6),
        'p29 top3=', round(summary['p29_top3_rate'], 6),
        'ranks=', summary['rank_counts'],
    )
pooled = next(row for row in validation['summaries'] if row['cohort'] == 'pooled')
assert pooled['phase_a_top1_rate'] == 5639/6000
assert pooled['phase_a_top2_rate'] == 5979/6000
assert pooled['phase_a_top3_rate'] == 5998/6000


low top1= 0.924 top2= 0.9965 top3= 0.9995 p29 top3= 0.745 ranks= {'1': 1848, '2': 145, '3': 6, '5': 1}
middle top1= 0.9405 top2= 0.996 top3= 1.0 p29 top3= 0.576 ranks= {'1': 1881, '2': 111, '3': 8}
high top1= 0.955 top2= 0.997 top3= 0.9995 p29 top3= 0.55 ranks= {'1': 1910, '2': 84, '3': 5, '4': 1}
pooled top1= 0.939833 top2= 0.9965 top3= 0.999667 p29 top3= 0.623667 ranks= {'1': 5639, '2': 340, '3': 19, '4': 1, '5': 1}


## Results — frozen decisions and controls

In [4]:
print(validation['registered_predictions'])
assert validation['registered_predictions']['P1_top1_at_least_90_percent'] is True
assert validation['registered_predictions']['P2_top2_at_least_99_percent'] is True
assert validation['registered_predictions']['P3_top3_at_least_99_9_percent'] is True
assert validation['registered_predictions']['P4_top3_beats_p29_by_50pp'] is False
assert validation['registered_predictions']['P5_frame_exact_zero_variance'] is True
assert validation['registered_predictions']['P6_reconstruction_and_truth_checks'] is True
print('Phase A top3 advantage over p29:', pooled['phase_a_top3_rate'] - pooled['p29_top3_rate'])
print('Phase A top3 advantage over odd scan:', pooled['phase_a_top3_rate'] - pooled['odd_top3_rate'])


{'P1_top1_at_least_90_percent': True, 'P2_top2_at_least_99_percent': True, 'P3_top3_at_least_99_9_percent': True, 'P4_top3_beats_p29_by_50pp': False, 'P5_frame_exact_zero_variance': True, 'P6_reconstruction_and_truth_checks': True}
Phase A top3 advantage over p29: 0.376
Phase A top3 advantage over odd scan: 0.7475


## Results — edge cases and frame

In [5]:
misses = [row for row in rows if int(row['phase_a_rank_of_prime']) > 3]
for row in misses:
    print(row['anchor'], '->', row['actual_next_prime'], 'rank', row['phase_a_rank_of_prime'])
assert len(misses) == 2
print(validation['cross_rung_frame'])
assert validation['cross_rung_frame']['value'] == 3.5
assert validation['cross_rung_frame']['variance'] == 0.0


71246886 -> 71246933 rank 5
710000379415 -> 710000379533 rank 4
{'value': 3.5, 'variance': 0.0, 'exact_all_rows': True, 'interpretation': 'scale/context coordinate only; it cannot rank targets because it is constant'}


## Takeaways

1. The corrected object is a complete child parent, not two individual factor labels.
2. Its first quiet state prospectively recovered 93.983% of next primes; two and three ranked states reached
   99.650% and 99.967%.
3. The result transferred across 71 million, 71 billion and 710 billion scales without target refitting.
4. The fixed `3.5` cross-rung route is an exact contextual frame, but its zero variance means it does not supply the
   changing prime correction.
5. Phase A remains a large partial sieve (780–48,817 children here). This is strong visible-state compression, not
   a three-operation or exact constant-cost prime algorithm.
6. The original validator failure remains recorded; v1.1 repaired only the child-table ceiling and reproduced all
   sealed predictions with 16/16 checks.
